# Ad spend and website visits: cross correlation by lag

**Cross Correlation** · a worked example from the StatNex teaching guide

**The story.** An online shop recorded its weekly ad spend and its website visits for ten weeks. Ads seem to bring visitors, but not straight away.

**The question.** Do website visits follow ad spend, and with what delay?

**To run it**, choose **Runtime → Run all**. The first cells save the table in this session. The rest is the calculation StatNex runs, written out in Python. It does not use StatNex, and every number it prints was checked against the app's.

## The table

This cell saves the table `08_time_series/cross_correlation.csv` (10 rows) as `data.csv`, the file the code reads.

In [ ]:
%%writefile data.csv
Week,Ad spend ($000),Website visits (thousands)
1.0,5.0,35.4
2.0,12.0,24.7
3.0,4.0,39.5
4.0,9.0,28.4
5.0,15.0,34.5
6.0,6.0,47.8
7.0,3.0,34.3
8.0,11.0,24.9
9.0,8.0,41.8
10.0,14.0,34.9

## Libraries

In [ ]:
import numpy as np
import pandas as pd
from statsmodels.tsa.stattools import ccf

pd.set_option("display.width", 120)

## The data

In [ ]:
# Two time series of the same length, in row order, with no blank cells.
data = pd.read_csv("data.csv", keep_default_na=False, na_values=[""])
x = data["Ad spend ($000)"].to_numpy(float)
y = data["Website visits (thousands)"].to_numpy(float)
n = len(x)
observations = pd.DataFrame({"No.": np.arange(1, n + 1), "Ad spend ($000)": x, "Website visits (thousands)": y})
print(observations.to_string(index=False))

## Cross-correlation

In [ ]:
# Does one series move first and the other follow? At lag k the first series is
# paired with the second k periods later:
#     r(k) = sum of (x(t) - mean x)(y(t + k) - mean y) / sqrt(Sxx * Syy)
# Each sum runs over the n - k pairs that overlap, but the means and the denominator
# use the whole series (the standard estimator, as R's ccf()), so r(k) shrinks a little
# as k grows. A peak at a positive lag means the first series leads; at a negative lag,
# the second leads. statsmodels' ccf(a, b, adjusted=False) pairs a(t + k) with b(t)
# for k = 0, 1, ..., so ccf(y, x) gives the positive lags and ccf(x, y) the negative
# ones. Lags up to n - 3 each way are kept.
max_lag = n - 3
ahead = ccf(y, x, adjusted=False, fft=False, nlags=max_lag + 1)   # lags 0, 1, ..., n - 3
behind = ccf(x, y, adjusted=False, fft=False, nlags=max_lag + 1)  # lags 0, -1, ..., -(n - 3)
cross = pd.DataFrame({"Lag": np.arange(-max_lag, max_lag + 1),
                      "Cross-correlation": np.concatenate([behind[:0:-1], ahead])})
print()
print(cross.to_string(index=False))
# A rough guide for one lag: a value beyond +/- 1.96 / sqrt(n) is hard to explain by
# chance -- if at least one of the series has no pattern of its own. When both zigzag
# or trend, big values arise by chance far more often, and picking the biggest of
# many lags makes it worse.
print(f"\nRough 95% band for one lag: +/- {1.96 / np.sqrt(n):.4f}")

## What the result says

Cross correlation measures how strongly two series move together when one is shifted in time. The lag is the size of the shift in weeks. A coefficient near 1 means a strong link in the same direction, and near 0 means no link.

At lag 0, the same week, the coefficient is -0.44: a big ad week is not a busy week. At lag 1 it is 0.86, by far the largest value in the table. So visits are high one week after ad spend is high. No other lag comes close.

Visits seem to follow ad spend with a delay of one week. But 10 weeks are few, and the far lags rest on only a few pairs of weeks. A link in time is also not proof that the ads cause the visits.

## Your own data

Open the **Files** panel (the folder icon on the left) and upload a CSV file. Then change `"data.csv"` in the `read_csv` line to its name, change the column names in the code to yours, and choose **Run all** again.